# Thí nghiệm 3: Áp dụng cải tiến kĩ thuật

**Mục tiêu:** Giữ nguyên model (YOLOv8n) và số epoch (50), chỉ áp dụng các kĩ thuật cải tiến để kiểm tra hiệu quả.

| Tham số | Baseline | Thí nghiệm này |
|---------|----------|-----------------|
| Model | YOLOv8n | YOLOv8n |
| Epochs | 50 | 50 |
| Image Size | 640 | 640 |
| Batch Size | 16 | 16 |
| Class Imbalance | **Không xử lý** | **Dùng `MixUp + Augmentation`** (tối ưu bộ nhớ) |
| Cosine LR | **Không** | **Có** |
| Early Stopping | patience=100 | **patience=15** |
| close_mosaic | 10 | **15** |
| degrees | 0.0 | **5.0** |
| scale | 0.5 | **0.3** |
| mosaic | 1.0 | **0.8** |
| mixup | 0.0 | 0.1 |

**Các kĩ thuật cải tiến áp dụng:**
1. **Thay thế Oversample Offline bằng `MixUp + Augmentation`** và thêm **mixup=0.1**: Giúp xử lý class thiểu số hiệu quả mà KHÔNG làm phình to dataset.
2. **Cosine LR Scheduler** — giảm learning rate mượt mà hơn
3. **Early Stopping** chặt hơn (patience=15) — tránh train thừa
4. **Tối ưu Augmentation** — giảm scale, thêm xoay nhẹ, giảm mosaic
5. **Tắt mosaic sớm hơn** (15 epoch cuối) để fine-tune

**Pipeline:** Clone repo → Tải data → Làm sạch → COCO→YOLO → Chia tập → Train → Đánh giá

In [1]:
!git clone https://github.com/Shiba-dotcom/waste-detection_project.git

Cloning into 'waste-detection_project'...
remote: Enumerating objects: 3743, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 3743 (delta 13), reused 12 (delta 12), pack-reused 3716 (from 2)
Receiving objects: 100% (3743/3743), 2.68 GiB | 47.68 MiB/s, done.
Resolving deltas: 100% (223/223), done.
Updating files: 100% (204/204), done.


In [2]:
!pip install gdown
# Tải file từ Drive về Kaggle
!gdown --id "17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM" -O raw.zip

# Chỉ cần tạo đến thư mục data (thư mục raw sẽ tự sinh ra khi giải nén)
!mkdir -p /kaggle/working/waste-detection_project/data

# Giải nén vào thư mục data
!unzip -q raw.zip -d /kaggle/working/waste-detection_project/data/

# Dọn dẹp file zip để tránh bị đầy bộ nhớ (Disk quota exceeded)
!rm raw.zip
!rm -rf /kaggle/working/waste-detection_project/.git


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM
From (redirected): https://drive.google.com/uc?id=17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM&confirm=t&uuid=6c08e6bb-4b1e-42f6-ad6d-05cf6593dc07
To: /kaggle/working/raw.zip
100%|██████████████████████████████████████| 2.62G/2.62G [00:32<00:00, 79.8MB/s]


In [3]:
%cd /kaggle/working/waste-detection_project
!pip install -r requirements.txt

/kaggle/working/waste-detection_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.1 MB/s eta 0:00:00a 0:00:01


In [4]:
# Di chuyển vào đúng thư mục của script rồi mới chạy
%cd /kaggle/working/waste-detection_project/src/data_prep
!python data_cleaning.py

%cd /kaggle/working/waste-detection_project/src
!python Training_dataYolo.py

%cd /kaggle/working/waste-detection_project/src/data_prep
!python split_dataset.py

/kaggle/working/waste-detection_project/src/data_prep
  DATA CLEANING - TACO Dataset

[Load] annotations.json ...
  So anh goc         : 1500
  So annotations goc : 4784
  So categories      : 60

────────────────────────────────────────────────────────────
[Buoc 1] Kiem tra dong trung lap (Duplicates)
────────────────────────────────────────────────────────────
  Duplicate annotations: 0
  Duplicate image IDs: 0
  Duplicate file names: 0

────────────────────────────────────────────────────────────
[Buoc 2] Kiem tra gia tri thieu (Missing Values)
────────────────────────────────────────────────────────────
  Images: Tat ca truong bat buoc day du
  Annotations: Tat ca truong bat buoc day du
  Anh khong co annotation: 0

────────────────────────────────────────────────────────────
[Buoc 3] Kiem tra nhan khong hop le
────────────────────────────────────────────────────────────
  Annotations voi category_id khong hop le: 0
  Categories khong co trong mapping.csv: 1
    - 'Rope & strings' 

### Bước huấn luyện

Sử dụng `MixUp + Augmentation` thay cho `oversample.py` để tiết kiệm dung lượng đĩa cứng trên Kaggle.

In [5]:
import os, time, glob
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

BASE_DIR = Path.cwd().parent.parent
YAML_PATH = BASE_DIR / "data/processed/dataset.yaml"
RESULTS = BASE_DIR / "results"
RESULTS.mkdir(exist_ok=True)

print("="*55)
print("  THI NGHIEM 3: YOLOv8n - 50 Epoch + Cai tien")
print("="*55)

model_path = BASE_DIR / "models/yolov8n.pt"
model = YOLO(str(model_path))
# Giữ nguyên YOLOv8n, 50 epoch — chỉ áp dụng cải tiến kĩ thuật
train_results = model.train(
    data     = str(YAML_PATH),
    epochs   = 50,           
    imgsz    = 640,
    batch    = 16,
    name     = "exp3_yolov8n_improved",
    project  = str(RESULTS / "runs"),
    exist_ok = True,
    verbose  = True,
    
    # --- CÁC THAM SỐ HUẤN LUYỆN CƠ BẢN ---
    patience     = 15,       # Early stopping
    cos_lr       = True,     # Cosine Annealing
    close_mosaic = 15,       # Tắt mosaic 15 epoch cuối để tinh chỉnh
    
    # --- TĂNG CƯỜNG DỮ LIỆU ĐỂ BÙ ĐẮP LỚP YẾU (CHẠY TRÊN RAM) ---
    mixup=0.15,      
    mosaic=0.8,      
    degrees=5.0,     # Xoay nhẹ 5 độ
    scale=0.3,       # Thu phóng 30%
    fliplr=0.5,      # Tỉ lệ lật ngang 50%
    hsv_h=0.015,     # Thay đổi sắc độ màu
    hsv_s=0.7,       # Thay đổi độ bão hòa
    hsv_v=0.4        # Thay đổi độ sáng
)

# Evaluation
print("\n[2] Danh gia tren tap Test ...")
metrics = model.val(split="test", verbose=False)

map50 = float(metrics.box.map50)
prec = float(metrics.box.mp)
recall = float(metrics.box.mr)
map5095 = float(metrics.box.map)

print(f"\nmAP@0.5    : {map50:.4f}")
print(f"mAP@0.5:0.95: {map5095:.4f}")
print(f"Precision  : {prec:.4f}")
print(f"Recall     : {recall:.4f}")

val_imgs = glob.glob(str(BASE_DIR / "data/processed/images/test/**/*.*"), recursive=True)[:50]
if val_imgs:
    t0 = time.perf_counter()
    for img_path in val_imgs:
        model.predict(img_path, verbose=False)
    elapsed = (time.perf_counter() - t0) / len(val_imgs) * 1000
    print(f"Inference  : {elapsed:.1f} ms/anh")
else:
    elapsed = None

result_row = {
    "Model": "YOLOv8n (50ep, cai tien)",
    "mAP@0.5 (%)": round(map50*100, 2),
    "mAP@0.5:0.95 (%)": round(map5095*100, 2),
    "Precision (%)": round(prec*100, 2),
    "Recall (%)": round(recall*100, 2),
    "Inference (ms)": round(elapsed, 1) if elapsed else "N/A",
}
df = pd.DataFrame([result_row])
df.to_csv(RESULTS / "ket_qua_exp3_improved.csv", index=False)
print("\nDa luu ket qua!")
print(df.to_string(index=False))

In [ ]:
# Nén kết quả để tải về
!zip -r /kaggle/working/exp3_yolov8n_improved_results.zip /kaggle/working/waste-detection_project/results/runs/exp3_yolov8n_improved
print("\nFile ZIP da san sang de tai ve!")